# CNN Training Fraction Ablation (Mumbi)

Trains the baseline CNN (TDSConvCTC, 32ch, 2000 Hz, log-spectrogram) at 25%, 50%, 75%, and 100% of available training data.

| Train Fraction | Val CER | Test CER |
|---|---|---|
| 25% | 28.2% | 30.1% |
| 50% | 23.0% | 24.9% |
| 75% | 21.1% | 22.2% |
| 100% | 18.9% | 21.2% |

> **GPU used:** NVIDIA L4 (Google Colab)

## 1. Setup

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

# ── Detect environment ─────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

env = 'Google Colab' if IN_COLAB else 'local Jupyter'
print(f'Environment: {env}')

# ── Repo root setup ────────────────────────────────────────────────────────
if IN_COLAB:
    REPO_ROOT = Path('/content/C247_mumbi_kai_jonathan_ben')
    if not REPO_ROOT.exists():
        subprocess.run(
            ['git', 'clone', 'https://github.com/Mumbzzz/C247_mumbi_kai_jonathan_ben',
             str(REPO_ROOT)],
            check=True
        )
else:
    # Walk up from cwd to find repo root (identified by setup.py)
    p = Path.cwd()
    while p != p.parent:
        if (p / 'setup.py').exists():
            REPO_ROOT = p
            break
        p = p.parent
    else:
        REPO_ROOT = Path.cwd()
        print('Warning: could not locate repo root, using cwd')

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f'Repo root: {REPO_ROOT}')

In [ ]:
import torch
print(f'CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -q protobuf==3.20.3 kenlm unidecode python-Levenshtein torchmetrics h5py==3.11.0
!pip install -q -e .

## 2. Configure Output Paths

In Colab, results and checkpoints are symlinked to Google Drive for persistence across sessions. Locally, they use repo-relative `results/` and `Playground_Mumbi/checkpoints/` directories.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    RESULTS_DIR     = Path('/content/drive/MyDrive/Classes/ECE247A/C247_results')
    CHECKPOINTS_DIR = Path('/content/drive/MyDrive/Classes/ECE247A/C247_checkpoints')
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

    for src, dst in [
        (str(RESULTS_DIR),     'results'),
        (str(CHECKPOINTS_DIR), 'Playground_Mumbi/checkpoints'),
    ]:
        dst_path = REPO_ROOT / dst
        if dst_path.is_symlink():  dst_path.unlink()
        elif dst_path.exists():    shutil.rmtree(dst_path)
        dst_path.symlink_to(src)

    r = (REPO_ROOT / 'results').resolve()
    c = (REPO_ROOT / 'Playground_Mumbi' / 'checkpoints').resolve()
    print(f'Results  → {r}')
    print(f'Checkpts → {c}')
else:
    RESULTS_DIR     = REPO_ROOT / 'results'
    CHECKPOINTS_DIR = REPO_ROOT / 'Playground_Mumbi' / 'checkpoints'
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Results  : {RESULTS_DIR}')
    print(f'Checkpts : {CHECKPOINTS_DIR}')

## 3. Download Data

Raw EMG data (~4.4 GB) is downloaded automatically in Colab. For local runs, download manually using the link printed below and place files in `data/89335547/`.

In [ ]:
DATA_DIR = REPO_ROOT / 'data' / '89335547'

if DATA_DIR.exists():
    print(f'Raw EMG data already present at {DATA_DIR}')
elif IN_COLAB:
    !wget -O 89335547.zip https://ucla.box.com/shared/static/dy28c8fne2d2bypyc0dz6xnfmmohgoyq
    !unzip 89335547.zip
    !mkdir -p data/89335547
    !mv 89335547/* data/89335547/
    !rm 89335547.zip
else:
    print('Raw EMG data not found.')
    print('Download: https://ucla.box.com/shared/static/dy28c8fne2d2bypyc0dz6xnfmmohgoyq')
    print(f'Extract and place files in: {DATA_DIR}')

## 4. CNN Training Fraction Ablation

Trains at 25%, 50%, 75%, and 100% of training data. Results saved to `results/results_summary_CNN_training_fraction_ablation.csv`.

In [ ]:
# Runs all 4 fractions sequentially
!python -m Playground_Mumbi.train \
    --model cnn \
    --epochs 80 \
    --batch-size 32 \
    --num-workers 2 \
    --run-all-fractions

In [ ]:
import pandas as pd
df = pd.read_csv('results/results_summary_CNN.csv')
print(df[['run_id', 'train_fraction', 'final_val_cer', 'test_cer']])

!python -m Playground_Mumbi.plot_results

if IN_COLAB:
    from google.colab import files
    files.download('results/results_summary_CNN.csv')
    files.download('results/results_curves_CNN.csv')
    files.download('Playground_Mumbi/checkpoints/final_models/best_cnn.pt')
    files.download('results/training_fraction_ablation_plots/cer_vs_fraction_CNN.png')
    files.download('results/training_fraction_ablation_plots/training_curves_CNN.png')
    files.download('results/training_fraction_ablation_plots/training_time_CNN.png')